# Generate synthetic KG using PyGraft from .data/mario/mario.tsv

To generate the synthetic KG I use a hand crafted set of files:
 - `.output/mario/schema.rdf`: Defines the schema of the graph.
 - `output/mario/class_info.json`: Defines the classes in the KG.
 - `output/mario/relation_info.json`: Defines the relations in the KG.

PyGraft names classes/relations generically (`C1`, `R1`...`R5`). The auto-generated schema was kept as [`generated_schema.rdf`](output/mario/generated_schema.rdf) for reference, then hand-customized into [`schema.rdf`](output/mario/schema.rdf) (along with matching [`class_info.json`](output/mario/class_info.json) and [`relation_info.json`](output/mario/relation_info.json), loaded directly by `generate_kg`):
- relations renamed to the real Mario ones (`brotherOf`, `servantOf`, `allyOf`, `enemyOf`, `loves`), with `rdfs:domain`/`rdfs:range` set to `Character` on all 5
- the one spurious `rdfs:subPropertyOf` dropped (a PyGraft library artifact, not a real characteristic of the data)
- the two placeholder classes replaced by `Character` (the class every entity in `mario.tsv` is typed as, via 14 `type` triples) plus an inert `PlaceholderClass` (PyGraft needs `num_classes >= 2` to run at all, but this second class is never assigned to any instance, since `avg_depth_specific_class: 1.0` and `prop_untyped_entities: 0.0` in `mario.yml` type every entity as exactly `Character`)

In [3]:
import pygraft

# pygraft.generate_kg("french_royalty.yml")
# pygraft.generate_kg("mario.yml")
pygraft.generate_kg("simple_mario.yml")



 _______                ______                        ___    _    
|_   __ \             .' ___  |                     .' ..]  / |_  
  | |__) |   _   __  / .'   \_|   _ .--.   ,--.    _| |_   `| |-' 
  |  ___/   [ \ [  ] | |   ____  [ `/'`\] `'_\ :  '-| |-'   | |   
 _| |_       \ '/ /  \ `.___]  |  | |     // | |,   | |     | |,  
|_____|    [\_:  /    `._____.'  [___]    \'-;__/  [___]    \__/  
            \__.'                                                 





Writing instance triples: 100%|██████████| 31/31 [00:00<00:00, 7135.91triples/s]



Consistent KG.



## Parse the result to a `.tsv` file

To compare against [`mario.tsv`](.data/mario/mario.tsv) (105 relation triples + 14 `type` triples), parse `full_graph.rdf` and keep only the entity-to-entity triples plus each entity's `rdf:type Character` triple, then serialize to Turtle/TSV.

In [4]:
from rdflib import RDF, Graph

def ttl2tsv(ttl_file, output_file):
    g = Graph()
    g.parse(ttl_file, format="turtle")

    with open(output_file, "w", encoding="utf-8") as f:
        n = 0
        for s, p, o in g:
            # Delete prefix if existing
            s = s.split("/")[-1].split("#")[-1]
            p = p.split("/")[-1].split("#")[-1]
            o = o.split("/")[-1].split("#")[-1]

            f.write(f"{s}\t{p}\t{o}\n")
            n += 1

    print(f"{n} triples written from {ttl_file} to {output_file}.")


def parse_result(full_graph, ttl_file, tsv_file, n_entities=15):
    g = Graph()
    g.parse(full_graph)

    ns = "http://pygraf.t/"
    entities = {f"{ns}E{i}" for i in range(1, n_entities)}
    character_class = f"{ns}Character"

    instances = Graph()
    for s, p, o in g:
        if str(s) not in entities:
            continue
        if str(o) in entities or (p == RDF.type and str(o) == character_class):
            instances.add((s, p, o))

    instances.serialize(ttl_file, format="turtle")
    print(f"{len(instances)} instance triples written from full_graph.rdf to {ttl_file}.")

    ttl2tsv(ttl_file, tsv_file)

In [1]:
ttl2tsv(ttl_file=".data/mario/simple_mario.ttl", output_file=".data/mario/simple_mario.tsv")

NameError: name 'ttl2tsv' is not defined

In [7]:
full_graph = "output/simple_mario/full_graph.rdf"
ttl_file = ".data/mario/simple_pygraft.ttl"
tsv_file = ".data/mario/simple_pygraft.tsv"

# full_graph = "output/mario/full_graph.rdf"
# ttl_file = ".data/mario/pygraft.ttl"
# tsv_file = ".data/mario/pygraft.tsv"

parse_result(full_graph=full_graph, ttl_file=ttl_file, tsv_file=tsv_file, n_entities=4429)

36 instance triples written from full_graph.rdf to .data/mario/simple_pygraft.ttl.
36 triples written from .data/mario/simple_pygraft.ttl to .data/mario/simple_pygraft.tsv.
